In [ ]:
from collections import Counter, OrderedDict, defaultdict
from abc import ABC, abstractmethod
from typing import Dict, Any, Sequence, Tuple, Optional, List

import numpy as np

import os
import sys

sources_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if sources_path not in sys.path:
    sys.path.append(sources_path)

import Common.config as config
import Common.datatypes as datatypes
import Common.utils as utils

import importlib

importlib.reload(config)
importlib.reload(datatypes)
importlib.reload(utils)

In [ ]:
class PrefetchScheduler:
    def __init__(
        self,
        *,
        R_M_D, 
        R_C_M, 
        U, 
        step_duration_s=1.0
    ):
        self.cfg = config.Config()

        self.R_M_D = float(R_M_D)  # bytes/s total across users
        self.R_C_M = float(R_C_M)  # bytes/s total across users
        self.U = int(U)
        self.tile_size_bytes = {
            0: 2.0e+6 / self.cfg.n_tiles,  # base layer tile size in bytes
            1: 1.5e+7 / self.cfg.n_tiles   # enhancement layer tile size in bytes
        }   # {0: size_base, 1: size_enh}
        self.step_duration_s = float(step_duration_s)

        self.now_s = 0.0  # simulation clock (sec)
        # availability[(cache_key, tile_key)] = ready_time_seconds
        self.availability: Dict[Tuple[str, Any], float] = {}

    @property
    def L_M_D_per_byte(self): 
        return 1.0 / (self.R_M_D / self.U)
    
    @property
    def L_C_M_per_byte(self): 
        return 1.0 / (self.R_C_M / self.U)

    def tick(self):
        self.now_s += self.step_duration_s

    @staticmethod
    def make_key(v, l, n, g):
        return (int(v), int(l), int(n), int(g))

    def tile_size(self, layer: int) -> float:
        return float(self.tile_size_bytes[layer])

    def is_scheduled(self, cache_key: str, tile_key):
        return (cache_key, tile_key) in self.availability

    def is_ready(self, cache_key: str, tile_key):
        rt = self.availability.get((cache_key, tile_key), None)
        return rt is not None and self.now_s >= rt

    # ---------------- MEC scheduling ----------------
    def schedule_to_mec(self, tile_key, size_bytes):
        mec_key = "MEC"
        if self.is_scheduled(mec_key, tile_key):
            return
        t_ready = self.now_s + size_bytes * self.L_C_M_per_byte
        self.availability[(mec_key, tile_key)] = t_ready

    # ---------------- DU scheduling -----------------
    def schedule_to_du(self, du_idx, tile_key, ensure_mec: bool, size_bytes):
        du_key = f"DU:{du_idx}"

        if self.is_scheduled(du_key, tile_key):
            return

        if ensure_mec:
            # Stage 1: Cloud -> MEC (if necessary)
            mec_key = "MEC"
            t_mec = self.availability.get((mec_key, tile_key), None)
            if t_mec is None or t_mec < self.now_s:
                t_mec = self.now_s + size_bytes * self.L_C_M_per_byte
                self.availability[(mec_key, tile_key)] = t_mec

            # Stage 2: MEC -> DU AFTER MEC is ready
            t_du = t_mec + size_bytes * self.L_M_D_per_byte
        else:
            # MEC is already planned -> only MEC->DU
            t_du = self.now_s + size_bytes * self.L_M_D_per_byte

        self.availability[(du_key, tile_key)] = t_du

    # ------------- Integration with planned bitmaps -------------
    def schedule_from_mec_plan(self, mec_bitmap_4d):
        if mec_bitmap_4d is None:
            return

        vs, ls, ns, gs = np.where(mec_bitmap_4d == 1)
        for v, l, n, g in zip(vs, ls, ns, gs):
            tk = self.make_key(v, l, n, g)
            size = self.tile_size(l)
            self.schedule_to_mec(tk, size)

    def schedule_from_du_plans(self, du_bitmaps_4d, mec_bitmap_4d=None):
        if not du_bitmaps_4d:
            return

        mec_planned = set()
        if mec_bitmap_4d is not None:
            vs, ls, ns, gs = np.where(mec_bitmap_4d == 1)
            mec_planned = {self.make_key(v, l, n, g) for v, l, n, g in zip(vs, ls, ns, gs)}

        for du_idx, du_bm in enumerate(du_bitmaps_4d):
            vs, ls, ns, gs = np.where(du_bm == 1)
            for v, l, n, g in zip(vs, ls, ns, gs):
                tk = self.make_key(v, l, n, g)
                size = self.tile_size(l)
                ensure_mec = tk not in mec_planned
                self.schedule_to_du(du_idx, tk, ensure_mec, size)

    # ------------- Produce "READY" bitmaps -----------------------
    def materialize_ready_bitmap(self, cache_key: str, planned_4d: np.ndarray):
        if planned_4d is None:
            return None
        ready = np.zeros_like(planned_4d, dtype=np.int8)
        vs, ls, ns, gs = np.where(planned_4d == 1)
        for v, l, n, g in zip(vs, ls, ns, gs):
            tk = self.make_key(v, l, n, g)
            if self.is_ready(cache_key, tk):
                ready[v, l, n, g] = 1
        return ready